In [ ]:
# PrandialTwin on Kaggle: Chronos-2 fine-tunes (plain and twin-informed) on CGMacros folds, ShanghaiT2DM (ext) and the
# six simulated Indian regions (plain only).
# Settings: Accelerator = GPU T4 x2 (one is used), Internet = On. Add Input: your PRIVATE dataset holding prandialtwin_kit.zip.
# Then "Save Version" -> "Save & Run All (Commit)"; it runs without the browser open. Download the out_<part>.zip files from Output.
# ext (ShanghaiT2DM) ran on 18 Oct. Split the rest over two accounts, e.g. "fold0 fold1 fold2 fold3 fold4" and the six india parts.
PARTS = "india_north india_south india_east india_west india_central india_northeast"

import glob, os, shutil, subprocess, zipfile
src = os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/twin/chronos_baseline.py", recursive=True)[0]))
shutil.copytree(src, "/kaggle/working/pt", dirs_exist_ok=True)  # inputs are read-only
# the data pickles were written with numpy 2.2 / pandas 2.3
subprocess.run("pip install -q chronos-forecasting==2.3.2 'numpy>=2.2,<2.3' 'pandas>=2.3,<2.4'", shell=True, check=True)

# One part at a time, zipped as soon as it is done, so a timeout loses only the part that was running.
# Kept: the forecasts and each trained model (finetuned-ckpt); not the trainer's duplicate checkpoint-N copies or the input data.
proc = "/kaggle/working/pt/data/processed"
models = {"ext": ["all15min", "tw_all15min"], **{f"fold{f}": [f"fold{f}", f"tw_fold{f}"] for f in range(5)},
          **{f"india_{r}": [f"india_{r}"] for r in ("north", "south", "east", "west", "central", "northeast")}}
for part in PARTS.split():
    # stream the run's output into this cell / the Log tab (a child process's prints do not show there otherwise)
    run = subprocess.Popen(f"python -u -m twin.chronos_baseline {part}", shell=True, cwd="/kaggle/working/pt",
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in run.stdout:
        print(line, end="", flush=True)
    assert run.wait() == 0, f"{part} failed: see the lines above"
    with zipfile.ZipFile(f"/kaggle/working/out_{part}.zip", "w", zipfile.ZIP_STORED) as z:
        for f in [f"{proc}/chronos_parts/{part}.pkl"] + sum([glob.glob(f"{proc}/models/chronos2_{m}/finetuned-ckpt/*") for m in models[part]], []):
            z.write(f, os.path.relpath(f, "/kaggle/working/pt"))
    shutil.rmtree(f"{proc}/models", ignore_errors=True)  # Kaggle's working copy only: this part's models are in its zip
    print(part, "zipped", flush=True)
shutil.rmtree("/kaggle/working/pt")
print("done:", os.listdir("/kaggle/working"))